In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

SEQ  = "RECORD@2020-11-21_13.46.26"   # city.  highway: "RECORD@2020-11-22_12.38.47"   countryside: "RECORD@2020-11-22_12.54.38"
TAG  = "city"                         # "highway" / "countryside"; goes into the saved figure name
DATA = rf"C:\Users\daksh\Documents\SPARSE_FFT\data\{SEQ}\{SEQ}"

log_path = os.path.join(DATA, SEQ + "_events_log.rec")
print(os.path.exists(log_path))       # must print True

In [ ]:
radar = {"radar_ch0": {}, "radar_ch1": {}, "radar_ch2": {}, "radar_ch3": {}}

with open(log_path) as f:
    for line in f:
        pieces = line.split()
        if len(pieces) != 17:
            continue
        sensor = pieces[10]
        if sensor not in radar:
            continue
        sample   = int(pieces[7])
        offset   = int(pieces[13])
        datasize = int(pieces[16])
        radar[sensor][sample] = (offset, datasize)

for name in radar:
    print(name, len(radar[name]))     # city: about 586-591 each, highway: about 488-493

In [ ]:
frames = []                           # ch3 sample numbers that have all 4 chips

for n in radar["radar_ch3"]:
    if (n + 1) in radar["radar_ch0"] and (n + 2) in radar["radar_ch2"] and (n + 3) in radar["radar_ch1"]:
        frames.append(n)

print(len(frames), frames[0], frames[-1])   # city: 576 frames, highway: 483

In [ ]:
def read_block(chip, offset, datasize):
    path = os.path.join(DATA, SEQ + f"_{chip}.bin")
    with open(path, "rb") as f:
        f.seek(offset)
        x = np.fromfile(f, dtype=np.int16, count=datasize//2)
    return x

def chip_cube(x):
    c = x[0::2] + 1j * x[1::2]
    c = np.reshape(c, (512, 4, 256), order="F")
    return c.transpose((0, 2, 1))

In [ ]:
# Hamming windows (frame_to_pred needs these)
N, M = 512, 256
w_range   = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(N) / (N - 1))
w_doppler = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(M) / (M - 1))

In [ ]:
from scipy.signal import convolve2d

def ca_cfar(P, n_rng=9, n_dop=9, g_rng=3, g_dop=3, thr_db=4):
    pw = np.abs(P) ** 2
    mask = np.ones((2*n_rng + 1, 2*n_dop + 1))
    mask[n_rng-g_rng : n_rng+g_rng+1, n_dop-g_dop : n_dop+g_dop+1] = 0

    sums   = convolve2d(pw, mask, mode="same")
    counts = convolve2d(np.ones_like(pw), mask, mode="same")
    noise  = sums / counts
    snr    = pw / noise
    return snr > 10 ** (thr_db / 10)

In [ ]:
def frame_to_pred(n, mti=False):
    keys = [("radar_ch3", n), ("radar_ch0", n+1), ("radar_ch1", n+3), ("radar_ch2", n+2)]

    cubes = []
    for chip, s in keys:
        off, size = radar[chip][s]
        cubes.append(chip_cube(read_block(chip, off, size)))   # each is (512, 256, 4)

    cube = np.concatenate(cubes, axis=2)                     # -> (512, 256, 16)
    if mti:
        cube = cube - cube.mean(axis=1, keepdims=True)
    else:
        cube = cube - cube.mean(axis=(0, 1), keepdims=True)

    rng = np.fft.fft(cube * w_range[:, None, None],   axis=0)
    rd  = np.fft.fft(rng  * w_doppler[None, :, None], axis=1)

    Pm = np.sum(np.abs(rd), axis=2)
    return Pm.reshape(512, 16, 16).sum(axis=1)